# PortPy bridge in pyCERR

This notebook demonstrates **`cerr.imrtp.portpy_bridge`** — the bridge that
exports pyCERR data as a [PortPy](https://github.com/PortPy-Project/PortPy)
dataset and drives PortPy's CVXPy **optimization**.

PortPy does **not** compute dose; it consumes a precomputed beamlet
dose-influence matrix `A` (`d = A x`) and optimizes fluence with clinical
criteria. pyCERR supplies `A` two ways:

| Source of `A` | Function | Needs pyRadPlan? |
|---|---|---|
| pyRadPlan pencil beam | `writePortpyFromPyRadPlan` | yes |
| pyCERR native **QIB** engine | `writePortpyFromQIB` | no |

**Requires:** `pip install portpy` (+ a CVXPy solver; free `SCS` is used here).

We walk through all four capabilities on a phantom:
1. Write a PortPy dataset (geometry) and load it back.
2. Attach a pyRadPlan influence matrix.
3. Attach a native-QIB influence matrix (no pyRadPlan).
4. Optimize with clinical criteria and import the dose.

## 0. Build a phantom `planC`

In [1]:
import numpy as np
import SimpleITK as sitk
from cerr import plan_container as pc
from cerr.imrtp import pyradplan_bridge as prp

def make_phantom(voxel_mm=4.0, shape_zyx=(28, 44, 44), radius_mm=14.0,
                 tmp_dir=None):
    """Water-box phantom with a BODY box and a central spherical PTV.

    Returns a pyCERR ``planC`` with scan 0 and structures BODY, PTV.
    """
    import tempfile, os
    nz, ny, nx = shape_zyx
    hu = np.full(shape_zyx, -1000.0, dtype=np.float32)   # air
    hu[3:-3, 6:-6, 6:-6] = 0.0                           # water box
    img = sitk.GetImageFromArray(hu)
    img.SetSpacing([voxel_mm] * 3)
    img.SetOrigin([0.0, 0.0, 0.0])
    tmp_dir = tmp_dir or tempfile.mkdtemp()
    nii = os.path.join(tmp_dir, "phantom.nii.gz")
    sitk.WriteImage(img, nii)
    planC = pc.loadNiiScan(nii, imageType="CT SCAN")

    scanImg = planC.scan[0].getSitkImage()
    def _import(mask_zyx, name):
        mImg = sitk.GetImageFromArray(mask_zyx.astype(np.uint8))
        mImg.CopyInformation(scanImg)
        m3 = prp.doseArrayFromSitk(mImg, planC, 0).astype(bool)
        pc.importStructureMask(m3, 0, name, planC)

    body = np.zeros(shape_zyx, dtype=bool)
    body[3:-3, 6:-6, 6:-6] = True
    _import(body, "BODY")
    cz, cy, cx = (nz - 1) / 2, (ny - 1) / 2, (nx - 1) / 2
    zz, yy, xx = np.ogrid[:nz, :ny, :nx]
    dist = voxel_mm * np.sqrt((zz-cz)**2 + (yy-cy)**2 + (xx-cx)**2)
    _import(dist <= radius_mm, "PTV")
    return planC

def struct_index(planC, name):
    return [i for i, s in enumerate(planC.structure)
            if s.structureName == name][0]

planC = make_phantom()
ptv = struct_index(planC, "PTV")
body = struct_index(planC, "BODY")
print("scan size:", tuple(planC.scan[0].getScanSize()),
      "| structures:", [s.structureName for s in planC.structure])

scan size: (np.int64(44), np.int64(44), np.int64(28)) | structures: ['BODY', 'PTV']


In [2]:
import matplotlib.pyplot as plt

def show_dose(planC, doseNum, scanNum=0, title="dose"):
    """Overlay a dose colourwash on the central CT slice."""
    scan3M = planC.scan[scanNum].getScanArray()
    dose3M = planC.dose[doseNum].doseArray
    k = scan3M.shape[2] // 2
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(scan3M[:, :, k], cmap="gray")
    m = ax.imshow(np.ma.masked_less(dose3M[:, :, k], 0.05 * dose3M.max()),
                  cmap="jet", alpha=0.5)
    plt.colorbar(m, ax=ax, label="Gy", fraction=0.046)
    ax.set_title("%s (slice %d)  max=%.2f Gy" % (title, k, dose3M.max()))
    ax.axis("off")
    plt.show()

## 1. Write a PortPy dataset and load it back

`writePortpyDataset` emits a PortPy-format patient folder (CT / Structures /
OptimizationVoxels / Beams as JSON + HDF5). PortPy's own `DataExplorer` loads
it unchanged.

In [3]:
import tempfile
import portpy.photon as pp
from cerr.imrtp import portpy_bridge as ppb

data_dir = tempfile.mkdtemp()
patDir = ppb.writePortpyDataset(
    planC, outDir=data_dir, patientId="Phantom",
    scanNum=0, structNums=[body, ptv], bodyStructNum=body,
    gantryAngles=[0.0, 72.0, 144.0, 216.0, 288.0], fieldHalfWidthMm=40.0)

data = pp.DataExplorer(data_dir=data_dir, patient_id="Phantom")
ct_pp = pp.CT(data)
structs = pp.Structures(data)
beams = pp.Beams(data)
print("PortPy loaded:  resolution", ct_pp.get_ct_res_xyz_mm(),
      "| structures", structs.get_structures(),
      "| beams", beams.get_all_beam_ids())

PortPy loaded:  resolution [4.0, 4.0, 4.0000000000000036] | structures ['BODY', 'PTV'] | beams [0, 1, 2, 3, 4]


## 2. Attach a pyRadPlan influence matrix

`writePortpyFromPyRadPlan` computes the influence matrix with pyRadPlan and
splits it into PortPy's per-beam sparse format on the dose grid.

In [4]:
from cerr.imrtp import pyradplan_bridge as prp

iso = prp.targetCentroidMm(planC, [ptv])
ct, cst, pln = prp.planFromPlanC(
    planC, scanNum=0, beamsNum=None,
    objectives={ptv: [prp.squaredDeviation(2.0)]},
    structNums=[ptv, body], targetStructNums=[ptv],
    gantryAngles=[0.0, 90.0], isoCenter=iso, bixelWidth=5.0,
    doseGridResolution={"x": 4.0, "y": 4.0, "z": 4.0})
stf, dij = prp.calcDoseInfluence(ct, cst, pln)

pr_dir = tempfile.mkdtemp()
ppb.writePortpyFromPyRadPlan(planC, dij, stf, outDir=pr_dir,
                             patientId="Phantom_PRP", scanNum=0,
                             structNums=[body, ptv], bodyStructNum=body,
                             isoCenter=iso)
d = pp.DataExplorer(data_dir=pr_dir, patient_id="Phantom_PRP")
inf = pp.InfluenceMatrix(pp.Structures(d), pp.Beams(d), pp.CT(d),
                         target_structure="PTV", is_bev=True)
print("pyRadPlan-sourced A:", inf.A.shape, "nnz", inf.A.nnz)

Beam:   0%|          | 0/2 [00:00<?, ?b/s]

Beam:   0%|          | 0/2 [00:00<?, ?b/s]

C:\Users\aptea\Miniconda3\envs\pycerr\Lib\site-packages\pyRadPlan\raytracer\_siddon.py:359: RuntimeWarning: divide by zero encountered in divide
  (p_min - self._source_points) / self._ray_vec,  # alpha to "near" plane
C:\Users\aptea\Miniconda3\envs\pycerr\Lib\site-packages\pyRadPlan\raytracer\_siddon.py:360: RuntimeWarning: divide by zero encountered in divide
  (p_max - self._source_points) / self._ray_vec,
C:\Users\aptea\Miniconda3\envs\pycerr\Lib\site-packages\numpy\lib\_function_base_impl.py:1496: RuntimeWarning: invalid value encountered in subtract
  a = op(a[slice1], a[slice2])
C:\Users\aptea\Miniconda3\envs\pycerr\Lib\site-packages\pyRadPlan\raytracer\_siddon.py:429: RuntimeWarning: invalid value encountered in multiply
  ijk = sp_scaled[:, :, None] + rv_scaled[:, :, None] * alphas_mid[:, None, :]


Ray:   0%|          | 0/77 [00:00<?, ?r/s]

Ray:  27%|██▋       | 21/77 [00:00<00:00, 189.29r/s]

...

Ray:  29%|██▊       | 22/77 [00:00<00:00, 219.37r/s]

Ray:  60%|█████▉    | 46/77 [00:00<00:00, 222.36r/s]

Ray:  90%|████████▉ | 69/77 [00:00<00:00, 214.42r/s]

Beam: 100%|██████████| 2/2 [00:02<00:00,  1.04b/s]

Creating BEV..
Loading sparse influence matrix...
Done
pyRadPlan-sourced A: (56144, 154) nnz 2166734


## 3. Attach a **native QIB** influence matrix (no pyRadPlan)

`writePortpyFromQIB` uses pyCERR's own QIB pencil-beam engine, so PortPy works
with no pyRadPlan installed.

In [5]:
from cerr.imrtp import imrtp_problem as imp
from cerr.imrtp.dosecalc import generateQIBInfluence

im = imp.initIMRTProblem(planC)
g = imp.addGoal(im, ptv, planC); g.isTarget = "yes"; g.xySampleRate = 1
imp.addEquispacedBeams(im, 2, 0.0, planC)
im.params.algorithm = "QIB"
for b in im.beams:
    imp.conditionBeam(b, im, planC)
generateQIBInfluence(im, planC)

qib_dir = tempfile.mkdtemp()
ppb.writePortpyFromQIB(planC, im, outDir=qib_dir, patientId="Phantom_QIB",
                       scanNum=0, structNums=[body, ptv], bodyStructNum=body)
d = pp.DataExplorer(data_dir=qib_dir, patient_id="Phantom_QIB")
inf_qib = pp.InfluenceMatrix(pp.Structures(d), pp.Beams(d), pp.CT(d),
                             target_structure="PTV", is_bev=True)
print("QIB-sourced A:", inf_qib.A.shape, "nnz", inf_qib.A.nnz)

Creating BEV..
Loading sparse influence matrix...
Done
QIB-sourced A: (160, 26) nnz 4025


## 4. Optimize with clinical criteria and import the dose

`optimizeAndImport` maps a prescription into a PortPy `opt_params` +
`ClinicalCriteria`, runs PortPy's `Optimization`, and writes the dose into
`planC.dose`. Here we optimize the QIB dataset.

In [6]:
sol, doseNum, planC = ppb.optimizeAndImport(
    planC, dataDir=qib_dir, patientId="Phantom_QIB",
    prescriptionGy=6.0, numFractions=3, scanNum=0,
    targetName="PTV", oarObjectives={"BODY": 0.0}, solver="SCS")
print("optimized; dose at planC.dose[%d]" % doseNum)
show_dose(planC, doseNum, title="PortPy optimized dose (QIB influence)")

Creating BEV..
Loading sparse influence matrix...
Done
Problem created
Running Optimization..
Optimal value: 36407.27591739733
Elapsed time: 0.062004804611206055 seconds
optimized; dose at planC.dose[0]


C:\Users\aptea\AppData\Local\Temp\ipykernel_32608\1263061914.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Notes & capabilities

- **Two interchangeable `A` sources** — pyRadPlan (physically modelled) or the
  native QIB engine (no extra dependency). Both produce a PortPy dataset that
  loads identically.
- **`opt_params` / clinical criteria** — `buildOptParams` and
  `buildClinicalCriteria` translate a prescription + OAR limits into PortPy's
  objective/criteria format. Edit them (or pass DVH/max-dose criteria) to
  reproduce a clinical protocol.
- **Solvers** — `SCS`/`ECOS`/`CLARABEL` are free; pass `solver="MOSEK"` (or
  Gurobi) once licensed for faster, tighter solves.
- **Grids** — the pyRadPlan path optimizes on the (coarser) dose grid; the QIB
  path uses the CT grid. Either way the dose is resampled back onto the scan
  grid on import.